## 1. Environment Setup & Dependencies

Install the required Python packages for running the notebook. We use `wandb` for potential logging and `tqdm` for progress bars.


In [1]:
# Cell 1 — Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric umap-learn wandb networkx tqdm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.8 MB/s eta 0:00:00


## 2. Repository Setup

Clone the GitHub repository to the local Colab environment and add `src` to the Python path.


In [2]:
# Cell 2 — Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')


Cloning into '/content/antenna-gnn'...
remote: Enumerating objects: 220, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 220 (delta 115), reused 186 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (220/220), 4.88 MiB | 11.54 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Repo ready at /content/antenna-gnn


## 3. Drive Mount and Path Configuration

Mount Google Drive to access `RAW_DATA` and store the computed `.pt` cache files in `DATA_ROOT`. This ensures our generated artifacts are preserved beyond the lifespan of the Colab session.


In [3]:
# Cell 3 — Mount Drive and set data paths
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed', f'{DATA_ROOT}/data/processed_finetune']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')


Mounted at /content/drive
Drive mounted. DATA_ROOT=/content/drive/MyDrive/antenna_gnn


## 4. PyG Graph Construction Logic

Core function to convert a raw antenna patch pattern and its S11 spectrum into a PyTorch Geometric `Data` object. This implements node feature engineering and virtual node connectivity.


In [4]:
import torch
from torch_geometric.data import Data
import numpy as np

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    # Node features: (N*N + 1) nodes, 5 features each
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])

    # Virtual global node (index N*N): all zeros except placeholder (is_seed=-1 for virtual node)
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)

    # 4-connectivity edges
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    # Virtual node edges (connect to all metal pixels only)
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]  # virtual edge type

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    # Target
    y = torch.tensor(s11_db, dtype=torch.float).unsqueeze(0)  # (1, 201)

    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)


## 5. Dataset Processing & Caching Loop

Iterate over the fine-tuning grid sizes. To overcome Google Drive's severe I/O latency when sequentially accessing thousands of files, we bulk-copy the raw `.mat` files to the local Colab disk using parallel workers before processing them.


In [5]:
import glob, os, torch, shutil, concurrent.futures
import scipy.io as sio
import numpy as np
from tqdm.auto import tqdm

for N in [35, 45, 55]:
    raw_files = sorted(glob.glob(f'{RAW_DATA}/fine-tuning dataset/{N}x{N}/**/Mat_Files/*.mat', recursive=True))
    seed_mask = np.load(f'{DATA_ROOT}/artifacts/seed_mask_{N}.npy')

    proc_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
    os.makedirs(proc_dir, exist_ok=True)

    print(f"Processing {N}x{N} grid, {len(raw_files)} files...")

    local_dir = f'/content/raw_{N}x{N}'
    os.makedirs(local_dir, exist_ok=True)

    print('Bulk copying files to local disk...')
    def copy_file(src):
        dst = os.path.join(local_dir, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        return dst

    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        local_paths = list(tqdm(executor.map(copy_file, raw_files), total=len(raw_files), desc=f'Copying {N}x{N} files'))

    functioning_count = 0
    for i, local_f in enumerate(tqdm(local_paths, desc=f'Processing {N}x{N} graphs')):
        proc_path = f'{proc_dir}/sample_{i}.pt'

        if os.path.exists(proc_path):
            data = torch.load(proc_path)
            functioning_count += getattr(data, 'is_functioning', 0)
        else:
            mat = sio.loadmat(local_f)
            is_functioning = int(mat['resonant_freqs'].size > 0)
            functioning_count += is_functioning

            data = build_pyg_graph(mat['patch_pattern'], mat['S11_dB'].flatten(), seed_mask, N)
            data.grid_size = N
            data.pixel_size_mm = 32.375 / N
            data.is_functioning = is_functioning
            torch.save(data, proc_path)

    print(f'Cleaning up local cache {local_dir}...')
    shutil.rmtree(local_dir)

    with open(f'{DATA_ROOT}/data/processed_finetune/{N}x{N}_DONE.txt', 'w') as fh:
        fh.write('DONE\n')

    print(f"Grid {N}x{N}: {len(raw_files)} total, {functioning_count/len(raw_files)*100:.1f}% functioning")


Processing 35x35 grid, 4988 files...
Bulk copying files to local disk...


Copying 35x35 files:   0%|          | 0/4988 [00:00<?, ?it/s]

Processing 35x35 graphs:   0%|          | 0/4988 [00:00<?, ?it/s]

Cleaning up local cache /content/raw_35x35...
Grid 35x35: 4988 total, 63.4% functioning
Processing 45x45 grid, 6984 files...
Bulk copying files to local disk...


Copying 45x45 files:   0%|          | 0/6984 [00:00<?, ?it/s]

Processing 45x45 graphs:   0%|          | 0/6984 [00:00<?, ?it/s]

Cleaning up local cache /content/raw_45x45...
Grid 45x45: 6984 total, 65.3% functioning
Processing 55x55 grid, 2992 files...
Bulk copying files to local disk...


Copying 55x55 files:   0%|          | 0/2992 [00:00<?, ?it/s]

Processing 55x55 graphs:   0%|          | 0/2992 [00:00<?, ?it/s]

Cleaning up local cache /content/raw_55x55...
Grid 55x55: 2992 total, 48.4% functioning


## 6. Stratified Data Splitting

Combine sample indices across all grids and perform a stratified 80/20 split based on `grid_size` and `is_functioning` to create the fine-tuning pool and held-out test set.


In [7]:
import json
from sklearn.model_selection import train_test_split
import torch_geometric.data # Import to make Data available for torch.load if using safe_globals

pool_list = []
labels_for_stratify = []

for N in [35, 45, 55]:
    proc_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
    pt_files = sorted(glob.glob(f'{proc_dir}/sample_*.pt'))

    for pt_file in tqdm(pt_files, desc=f"Loading metadata for {N}x{N} split"):
        idx = int(os.path.basename(pt_file).split('_')[1].split('.')[0])
        data = torch.load(pt_file, weights_only=False)
        pool_list.append((N, idx))
        labels_for_stratify.append(f"{N}_{data.is_functioning}")

train_pool, test_set = train_test_split(
    pool_list, test_size=0.2, stratify=labels_for_stratify, random_state=42
)

with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'w') as f:
    json.dump(test_set, f)

with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'w') as f:
    json.dump(train_pool, f)

print(f"Total pool size: {len(train_pool)}")
print(f"Total test size: {len(test_set)}")

for name, dataset in [("Pool", train_pool), ("Test", test_set)]:
    print(f"\n{name} set breakdown:")
    for N in [35, 45, 55]:
        count = sum(1 for item in dataset if item[0] == N)
        print(f"  {N}x{N}: {count}")

Loading metadata for 35x35 split:   0%|          | 0/4988 [00:00<?, ?it/s]

Loading metadata for 45x45 split:   0%|          | 0/6984 [00:00<?, ?it/s]

Loading metadata for 55x55 split:   0%|          | 0/2992 [00:00<?, ?it/s]

Total pool size: 11971
Total test size: 2993

Pool set breakdown:
  35x35: 3990
  45x45: 5587
  55x55: 2394

Test set breakdown:
  35x35: 998
  45x45: 1397
  55x55: 598


## 7. Normalization Statistics

### Why Reuse Normalization Statistics?
It is **CRITICAL** to use the exact same `s11_mean.npy` and `s11_std.npy` computed from the 25x25 training set in Chunk 5. Do **NOT** recompute new normalization stats from this fine-tuning dataset. The pretrained model's output layer expects targets normalized under the original 25x25 distribution. Changing the normalization here would invalidate the pretrained weights and silently break the transfer learning initialization.


In [8]:
import numpy as np

# Load s11_mean and s11_std computed from the 25x25 training set
s11_mean = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
s11_std  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')

print(f"s11_mean shape: {s11_mean.shape}")
print(f"s11_std shape: {s11_std.shape}")


s11_mean shape: (201,)
s11_std shape: (201,)


## 8. Verification & DataLoader Test

Verify that the DataLoader outputs properly batched graph objects with mixed grid sizes, and spot check the fixed test set.


In [10]:
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader

class FinetuneDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        grid_size, local_idx = self.indices[idx]
        return torch.load(f'{DATA_ROOT}/data/processed_finetune/{grid_size}x{grid_size}/sample_{local_idx}.pt', weights_only=False)

with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'r') as f:
    pool_indices = json.load(f)

pool_dataset = FinetuneDataset(pool_indices)
loader = DataLoader(pool_dataset, batch_size=16, shuffle=True)

batch = next(iter(loader))
print("Batch from Pool:")
print(batch)
print("Grid sizes in this batch:", batch.grid_size.tolist())
print("NaNs in x:", torch.isnan(batch.x).any().item())
print("NaNs in y:", torch.isnan(batch.y).any().item())

print("\nSpot check FIXED test set:")
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'r') as f:
    test_indices = json.load(f)

test_dataset = FinetuneDataset(test_indices)
for i in range(5):
    sample = test_dataset[i]
    print(f"Sample {i}: grid_size={sample.grid_size}, is_functioning={sample.is_functioning}")

Batch from Pool:
DataBatch(x=[32416, 5], edge_index=[2, 163250], edge_attr=[163250, 2], y=[16, 201], grid_size=[16], pixel_size_mm=[16], is_functioning=[16], batch=[32416], ptr=[17])
Grid sizes in this batch: [55, 35, 45, 45, 35, 45, 45, 35, 55, 35, 45, 55, 45, 55, 45, 35]
NaNs in x: False
NaNs in y: False

Spot check FIXED test set:
Sample 0: grid_size=45, is_functioning=0
Sample 1: grid_size=45, is_functioning=1
Sample 2: grid_size=35, is_functioning=1
Sample 3: grid_size=55, is_functioning=1
Sample 4: grid_size=45, is_functioning=1
